# SIDInspector quickstart

This notebook runs the CPU demo on the small music-like mapping in `examples/reviewer_quickstart_data/`. It normalizes an exported item-to-SID table, validates the joins, and prints D1–D5 profiles.

The demo is a usability check. It does not reproduce the CIKM 2026 paper tables, and it does not train a tokenizer.

From a checkout of this repository:

```bash
python3 -m pip install -e .
```

The package is not published on PyPI. Open the notebook with its working directory at the repository root, or anywhere inside the checkout.

In [ ]:
from pathlib import Path
import subprocess
import sys

import pandas as pd

import sidinspector


def repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        marker = candidate / "examples" / "reviewer_quickstart_data" / "sid_codes.csv"
        if marker.is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find the SIDInspector checkout. Open this notebook from the repository."
    )


ROOT = repo_root()
DATA = ROOT / "examples" / "reviewer_quickstart_data"
print(sidinspector.__name__, "from", ROOT)
pd.read_csv(DATA / "sid_codes.csv").head()

In [ ]:
out = ROOT / "examples" / "reviewer_quickstart_output" / "notebook"
subprocess.run(
    [sys.executable, str(ROOT / "examples" / "run_reviewer_quickstart.py"), "--output-dir", str(out)],
    cwd=ROOT,
    check=True,
)
diagnostics = out / "diagnostics"
d2 = pd.read_csv(diagnostics / "d2_collision.csv")
d3 = pd.read_csv(diagnostics / "d3_alignment.csv")
d5 = pd.read_csv(diagnostics / "d5a_deployment_cost.csv")
d2_full = d2[d2["prefix_depth"] == 3].iloc[0]
d3_l1 = d3[d3["prefix_depth"] == 1].iloc[0]
print(f"preflight: {out / 'preflight_summary.json'}")
print(f"D2 full aliasing rate: {d2_full['full_collision_rate']:.3f}")
print(f"D3 depth-1 weighted co-occurrence recall: {d3_l1['weighted_collab_prefix_recall']:.3f}")
print(f"D5 active prefix counts: {d5.iloc[0]['prefix_counts']}")
d2

In [ ]:
from IPython.display import display

d1 = pd.read_csv(diagnostics / "d1_utilization.csv")
d4 = pd.read_csv(diagnostics / "d4_head_tail.csv")
display(d1)
display(d3)
display(d4)
display(d5)